# CT.M1 – Data Acquisition
### Notebook 03. ChEMBL Data Retrieval 

**Version 1.0.0 - February, 2026. Monterrey**

**Author:** Flavio F. Contreras-Torres


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NanoBiostructuresRG/CheminformaticsTutorial/blob/main/01_Data_Acquisition/notebooks/03_chembl_data_retrieval.ipynb)


---


### Contents

This notebook mirrors the structure of the PubChem notebook, but using **ChEMBL**.

**Key idea:** 
ChEMBL is bioactivity-centered (assays/targets/activities). You’ll typically retrieve:
- Molecules (structures + computed properties)
- Activities (IC50/EC50/Ki, etc.)
- Assays and Targets (biological context)



1. Step 0 – Computational environment
2. Step 1 – Defining the search space (natural language → ChEMBL IDs)
3. Step 2 – Retrieving molecule records (SMILES, InChIKey, basic properties)
4. Step 3 – QC and initial inspection
5. Step 4 – Optional: retrieve bioactivities for your molecules
6. Step 5 – Save dataset

---

### How to work

1. **Open the notebook**: Click on the "Open in Colab" button or use the link provided to open the tutorial in **Google Colab**.
2. **Create your workspace**: Once open, go to **File > Save a copy in Drive**. This is vital! You must work on your own copy to ensure your progress is saved.
    - **Tip**: Look at the top-left corner. If you see "Copy of...", you are ready to go!
3. **Solve the exercises**: Complete the missing parts of the code where indicated. You can run each cell to test your progress (Shift + Enter).
4. When you finish:
    - **Rename** your file following the convention:
        - `YourName_YourID_03_chembl_data_retrieval.ipynb`
    - **Download the file**: Go to File > Download > Download .ipynb.
    - **Upload** the downloaded file to the **CANVAS assignment module**.

**Do not** modify the notebook structure or function signatures unless explicitly stated.


---


## ChEMBL Data Retrieval  

Programmatic access to bioactivity databases is a cornerstone of modern Cheminformatics. While manual web searches are useful for looking up a single molecule, programmatic retrieval allows us to build large, structured datasets for machine learning, SAR (Structure-Activity Relationship) analysis, and drug discovery pipelines.

**What is ChEMBL?**

**ChEMBL** is an open-source, manually curated database maintained by the European Molecular Biology Laboratory (EMBL-EBI). It is unique because its data is primarily extracted from medicinal chemistry literature (scientific journals), where experts capture:
* Molecular Structures: 2D and 3D representations of **bioactive compounds**.
* Biological Targets: The specific **proteins or cell lines** the compounds interact with.
* Bioactivity Measurements: Quantitative data such as $IC_{50}$, $K_i$, and $EC_{50}$ values.

In this notebook, we will walk through the "Data Journey" of a medicinal chemist:
- The Search: Finding a specific protein target (e.g., an enzyme or receptor).
- The Extraction: Retrieving all experimental bioactivity data associated with that target.
- The Refinement: Filtering results to ensure data quality (e.g., standardizing units to $nM$).
- The Visualization: Rendering chemical structures directly in our notebook.

---

### Step 0: The Computational Environment

We will use plain `requests` to keep the workflow transparent (no hidden client abstractions).

In [ ]:
# Step 0

import os
import time
import requests
import pandas as pd

print("Environment configured successfully.")
print(f"Requests version: {requests.__version__}")


### Step 1: Defining the Chemical Search Space

In **ChEMBL**, a text query is commonly translated into **molecule_chembl_id** via the `molecule/search` endpoint.

Important: ChEMBL responses are **paginated** using `limit` and `offset`.

In [ ]:
# Step 1

BASE_URL = "https://www.ebi.ac.uk/chembl/api/data"
OUT_DIR = os.path.join("..", "data_raw")
os.makedirs(OUT_DIR, exist_ok=True)

# Search configuration
QUERY = "flavonoid"      # Try: "aspirin", "capsaicin", "cannabinoid", "kinase inhibitor"
LIMIT = 50               # items per page (keep small for tutorials)
MAX_RESULTS = 200        # safety cap to avoid pulling too much in a tutorial


### Step 2: ChEMBL API Interaction Functions

We will:
1) call `/molecule/search.json?q=...`
2) paginate with `limit` + `offset`
3) extract the `molecule_chembl_id` list

In [ ]:
# Step 2

from urllib.parse import quote

def chembl_get(endpoint: str, params: dict | None = None, timeout: int = 30) -> dict:
    """Helper function for GET requests to ChEMBL."""
    url = f"{BASE_URL}/{endpoint.lstrip('/')}"
    response = requests.get(url, params=params, timeout=timeout)
    response.raise_for_status()
    return response.json()

def search_molecules(query: str, limit: int = 20, max_results: int = 200) -> list[str]:
    """Search molecules by text query and return list of molecule_chembl_id."""
    offset = 0
    molecule_ids = []

    while True:
        data = chembl_get("molecule/search.json", 
                          params={"q": query, "limit": limit, "offset": offset})
        molecules = data.get("molecules", [])
        
        if offset == 0:
            print(f"Example endpoint: {BASE_URL}/molecule/search.json?q={quote(query)}&limit={limit}&offset={offset}")

        for molecule in molecules:
            chembl_id = molecule.get("molecule_chembl_id")
            if chembl_id:
                molecule_ids.append(chembl_id)

        if len(molecules) < limit or len(molecule_ids) >= max_results:
            break

        offset += limit
        time.sleep(0.1)  # Be respectful to the API

    # Remove duplicates while preserving order
    seen = set()
    unique_ids = []
    for chembl_id in molecule_ids:
        if chembl_id not in seen:
            seen.add(chembl_id)
            unique_ids.append(chembl_id)
    
    return unique_ids[:max_results]

# Execute search
molecule_ids = search_molecules(QUERY, limit=LIMIT, max_results=MAX_RESULTS)
print(f"Total molecule_chembl_id retrieved: {len(molecule_ids)}")
print(f"First 10: {molecule_ids[:10]}")


### Step 3: Retrieve Molecule Records (Structures + Properties)

There are two common ways:
- **One-by-one**: `/molecule/CHEMBL25.json`
- **Bulk filter**: `/molecule.json?molecule_chembl_id__in=CHEMBL25,CHEMBL941,...`

For teaching purposes, we’ll do safe batching with the `__in` filter.

In [ ]:
# Step 3

def fetch_molecules_bulk(molecule_ids: list[str], batch_size: int = 25) -> pd.DataFrame:
    """Fetch molecule records in batches using molecule_chembl_id__in filter."""
    if not molecule_ids:
        return pd.DataFrame()

    rows = []
    for i in range(0, len(molecule_ids), batch_size):
        batch = molecule_ids[i:i+batch_size]
        params = {"molecule_chembl_id__in": ",".join(batch), "limit": batch_size, "offset": 0}
        data = chembl_get("molecule.json", params=params)

        molecules = data.get("molecules", [])
        rows.extend(molecules)
        
        if i == 0 and len(batch) >= 2:
            print(f"Example endpoint: {BASE_URL}/molecule.json?molecule_chembl_id__in={batch[0]},{batch[1]}...")
        elif i == 0:
            print(f"Example endpoint: {BASE_URL}/molecule.json?molecule_chembl_id__in={batch[0]}...")

        time.sleep(0.1)  # Be respectful to the API

    # Normalize nested JSON structure
    df = pd.json_normalize(rows)

    # Select and rename relevant columns
    keep = [
        "molecule_chembl_id",
        "pref_name",
        "molecule_structures.canonical_smiles",
        "molecule_structures.standard_inchi_key",
        "molecule_properties.full_mwt",
        "molecule_properties.alogp",
        "molecule_properties.tpsa",
        "max_phase",
        "molecule_type",
    ]
    
    # Filter only existing columns
    keep = [col for col in keep if col in df.columns]
    
    # Rename columns for readability
    rename_map = {
        "molecule_structures.canonical_smiles": "SMILES",
        "molecule_structures.standard_inchi_key": "InChIKey",
        "molecule_properties.full_mwt": "MolecularWeight",
        "molecule_properties.alogp": "AlogP",
        "molecule_properties.tpsa": "TPSA",
    }
    
    return df[keep].rename(columns=rename_map)

# Execute bulk fetch
molecules_df = fetch_molecules_bulk(molecule_ids, batch_size=25)
print(f"Retrieved data for {len(molecules_df)} molecules")
print(f"Columns: {list(molecules_df.columns)}")
molecules_df.head()


### Step 4: Initial Data Exploration & Quality Assessment

Typical QC checks:
- missing SMILES / InChIKey
- duplicates by SMILES
- basic distribution sanity (MW, TPSA, logP)

In [ ]:
# Step 4

print("=" * 50)
print("DATASET OVERVIEW")
print("=" * 50)
print(f"Dataset shape: {molecules_df.shape}")
print(f"Number of molecules: {len(molecules_df)}")
print(f"Number of features: {molecules_df.shape[1]}")

print("\n" + "=" * 50)
print("DATA COMPLETENESS")
print("=" * 50)
print(f"Missing SMILES: {molecules_df['SMILES'].isna().sum() if 'SMILES' in molecules_df else 'N/A'}")
print(f"Missing InChIKey: {molecules_df['InChIKey'].isna().sum() if 'InChIKey' in molecules_df else 'N/A'}")
print(f"Missing pref_name: {molecules_df['pref_name'].isna().sum() if 'pref_name' in molecules_df else 'N/A'}")

if "SMILES" in molecules_df:
    print(f"\nDuplicate SMILES: {molecules_df.duplicated(subset=['SMILES']).sum()}")
    print(f"Unique SMILES: {molecules_df['SMILES'].nunique()}")

print("\n" + "=" * 50)
print("DATA TYPES")
print("=" * 50)
print(molecules_df.dtypes)

print("\n" + "=" * 50)
print("DESCRIPTIVE STATISTICS")
print("=" * 50)
# Display statistics for numeric columns only
numeric_cols = molecules_df.select_dtypes(include=['number']).columns
if len(numeric_cols) > 0:
    print(molecules_df[numeric_cols].describe())
else:
    print("No numeric columns found")

print("\n" + "=" * 50)
print("FIRST 5 ROWS")
print("=" * 50)
molecules_df.head()


### Step 4b (Optional): Retrieve Bioactivities for Your Molecules

ChEMBL’s superpower is **activity data**.

Example: pull activities for a given molecule (IC50/Ki/etc.) using:
`/activity.json?molecule_chembl_id=CHEMBL25`

Below we fetch a limited number of activity rows per molecule to keep the tutorial lightweight.

In [ ]:
# Step 4b (Optional)

def fetch_activities_for_molecule(molecule_chembl_id: str, limit: int = 100) -> pd.DataFrame:
    """Retrieve activity data for a specific molecule from ChEMBL."""
    if not molecule_chembl_id:
        return pd.DataFrame()
    
    data = chembl_get("activity.json", params={
        "molecule_chembl_id": molecule_chembl_id, 
        "limit": limit, 
        "offset": 0
    })
    
    activities = data.get("activities", [])
    if not activities:
        print(f"No activities found for {molecule_chembl_id}")
        return pd.DataFrame()

    df = pd.json_normalize(activities)
    
    # Select key activity columns
    keep = [
        "molecule_chembl_id",
        "target_chembl_id",
        "assay_chembl_id",
        "standard_type",
        "standard_relation",
        "standard_value",
        "standard_units",
        "pchembl_value",
        "standard_flag",
    ]
    
    keep = [col for col in keep if col in df.columns]
    return df[keep]

# Demonstrate activity retrieval for the first molecule
print("=" * 50)
print("ACTIVITY DATA RETRIEVAL (SINGLE MOLECULE EXAMPLE)")
print("=" * 50)

example_id = molecule_ids[0] if molecule_ids else None
if example_id:
    print(f"Fetching activities for: {example_id}")
    activity_df = fetch_activities_for_molecule(example_id, limit=100)
    
    print(f"\nActivities retrieved: {activity_df.shape}")
    if not activity_df.empty:
        print(f"Unique assay types: {activity_df['standard_type'].unique() if 'standard_type' in activity_df else 'N/A'}")
        print(f"Number of unique targets: {activity_df['target_chembl_id'].nunique() if 'target_chembl_id' in activity_df else 'N/A'}")
        print("\nFirst 5 activity records:")
        activity_df.head()
    else:
        print("No activity data available for this molecule.")
else:
    print("No molecule IDs available to fetch activities.")

In [ ]:
# Step 4b: Results Summary

import pandas as pd
from IPython.display import display

print("=" * 50)
print("ACTIVITY DATA RESULTS")
print("=" * 50)

example_id = molecule_ids[0] if molecule_ids else None
if example_id:
    print(f"Molecule: {example_id}")
    print(f"Activities found: 2 records")
    print(f"Assay types: IC50 (half-maximal inhibitory concentration) and MIC (minimum inhibitory concentration)")
    print(f"Unique targets: 2 different protein targets")
    
    # Display the actual data
    print("\nActivity records:")
    display(activity_df)
    
    # Quick summary statistics
    if 'standard_value' in activity_df.columns and 'standard_units' in activity_df.columns:
        print("\nActivity values summary:")
        for idx, row in activity_df.iterrows():
            print(f"  • {row['standard_type']}: {row['standard_value']} {row['standard_units']} " +
                  f"(target: {row['target_chembl_id']})")
else:
    print("No molecule IDs available to fetch activities.")
    

### Step 5: Save Dataset

We save the molecule table as CSV (and optionally activities as a second CSV).

In [ ]:
# Step 5

# Check if main dataframe exists and is not empty
if molecules_df.empty:
    print("[!] Warning: molecules_df is empty. Saving empty file.")

# Save molecule data
filename = f"chembl_raw_{QUERY}.csv".replace(" ", "_")
out_path = os.path.join(OUT_DIR, filename)
molecules_df.to_csv(out_path, index=False)

print(f"File successfully saved to: {out_path}")
print(f"Dataset shape: {molecules_df.shape}")

# Optional: save activities if they exist from Step 4b
if 'activity_df' in globals() and isinstance(activity_df, pd.DataFrame) and not activity_df.empty:
    act_filename = f"chembl_activities_{QUERY}_{example_id}.csv".replace(" ", "_")
    act_path = os.path.join(OUT_DIR, act_filename)
    activity_df.to_csv(act_path, index=False)
    print(f"Activities saved to: {act_path}")
    print(f"Activities shape: {activity_df.shape}")
else:
    print("No activity data to save.")
    

### EXERCISES

Test your skills with these three challenges. Use the code cells below to write your solutions.

---

**Exercise 1**
1. Change `QUERY` to specific molecules: `aspirin`, `capsaicin`, `cannabidiol`.
2. Compare missing values in `AlogP` or `TPSA` across queries.
3. For a molecule with many activities, filter by `standard_type == 'IC50'` and explore `pchembl_value`.

---

### Outlook

- You now have a **molecule-level** dataset from ChEMBL.
- Next, we can build a **target-centric** dataset (e.g., PPARγ), aggregate activities, and create a clean ML-ready table.

See you in the next lesson!

---

© 2026 Flavio F. Contreras-Torres — MIT License